# **Environmental Time-Series Intelligence System**
### A Forecasting & Risk Intelligence Pipeline for Urban Air Quality

---

| | |
|---|---|
| **Author** | Victor Sunarko |
| **Date** | 12th March 2026 |
| **Version** | 1.0 |

---

## Dataset Citations

**Primary:** Zhang, S., Guo, B., Dong, A., He, J., Xu, Z., & Chen, S.X. (2017).  
*Cautionary Tales on Air-Quality Improvement in Beijing.*  
Proceedings of the Royal Society A, 473(2205), 20170457.  
UCI ML Repository ID: 501 — Beijing Multi-Site Air Quality Data  

**Extension:** OpenAQ Platform — Jakarta Station Data (2022–2025)  
Source: https://openaq.org | License: CC BY 4.0  

---

## Abstract

This project constructs a comprehensive environmental forecasting and risk intelligence
system applied to urban air quality in Beijing, China. The pipeline integrates temporal
data validation, statistical inference, classical time-series modeling (ARIMA, SARIMA,
SARIMAX, GARCH), machine learning forecasting (Random Forest, XGBoost, LightGBM),
uncertainty quantification, model interpretability (SHAP), regime analysis, and a
practical AQI-based risk intelligence framework. A geographic extension applies the
methodology to Jakarta, Indonesia using real-time OpenAQ API data. All modeling
decisions are assumption-tested, all forecasts are interval-accompanied, and all
conclusions are interpreted in their environmental and policy context.

---

> *"A forecast without uncertainty is not a forecast — it is an assertion."*



# **CHAPTER 0: Project Initialization & Reproducibility**


In [2]:
# CELL 0.2 — Library Imports
# Grouped by functional purpose for readability
# ============================================================

# --- Core ---
import os
import warnings
import json
import joblib
from datetime import datetime
from pathlib import Path

# --- Data Manipulation ---
import numpy as np
import pandas as pd

# --- Statistical Tests & Models ---
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, ccf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from arch.unitroot import ADF, KPSS

# --- Machine Learning ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              precision_score, recall_score, f1_score,
                              classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import xgboost as xgb
import lightgbm as lgb

# --- Interpretability ---
import shap

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# --- Data Acquisition ---
from ucimlrepo import fetch_ucirepo
import requests

# --- Utilities ---
from tqdm import tqdm
from itertools import product

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")

All libraries imported successfully.


In [4]:
# CELL 0.3 — Global Configuration & Reproducibility Settings
# All random states, plot styles, and display options
# set here once and inherited throughout the notebook
# ============================================================

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Plot Style Configuration ---
plt.rcParams.update({
    'figure.figsize'      : (14, 5),
    'figure.dpi'          : 120,
    'axes.spines.top'     : False,
    'axes.spines.right'   : False,
    'axes.grid'           : True,
    'grid.alpha'          : 0.3,
    'grid.linestyle'      : '--',
    'font.family'         : 'DejaVu Sans',
    'axes.titlesize'      : 13,
    'axes.titleweight'    : 'bold',
    'axes.labelsize'      : 11,
    'xtick.labelsize'     : 9,
    'ytick.labelsize'     : 9,
    'legend.fontsize'     : 9,
    'legend.framealpha'   : 0.7,
})

PALETTE = {
    'primary'   : '#2C7BB6',
    'secondary' : '#D7191C',
    'tertiary'  : '#1A9641',
    'warning'   : '#FDAE61',
    'neutral'   : '#636363',
    'light'     : '#F0F0F0',
}

# --- Pandas Display Options ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 120)

# --- Project Constants ---
TARGET_VARIABLE     = 'PM2.5'
FORECAST_HORIZON    = 24          # hours ahead
TRAIN_RATIO         = 0.70
VAL_RATIO           = 0.15
TEST_RATIO          = 0.15
AQI_THRESHOLDS      = {
    'Good'          : (0,   12),
    'Moderate'      : (12,  35.4),
    'Unhealthy_SG'  : (35.4, 55.4),
    'Unhealthy'     : (55.4, 150.4),
    'Very_Unhealthy': (150.4, 250.4),
    'Hazardous'     : (250.4, float('inf')),
}
WHO_THRESHOLD       = 15.0        # WHO 24h mean guideline (µg/m³)

print("Global configuration set.")
print(f"   Target variable   : {TARGET_VARIABLE}")
print(f"   Forecast horizon  : {FORECAST_HORIZON} hours")
print(f"   Train/Val/Test    : {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")
print(f"   Random seed       : {RANDOM_SEED}")
print(f"   WHO threshold     : {WHO_THRESHOLD} µg/m³")

Global configuration set.
   Target variable   : PM2.5
   Forecast horizon  : 24 hours
   Train/Val/Test    : 0.7/0.15/0.15
   Random seed       : 42
   WHO threshold     : 15.0 µg/m³


In [6]:
# CELL 0.4 — Directory Structure Creation
# Creates all project folders if they do not already exist
# ============================================================

DIRS = [
    'data/raw/beijing',
    'data/raw/jakarta',
    'data/processed',
    'models/saved_models',
    'outputs/figures',
    'outputs/tables',
]

for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Project directory structure created:")
for d in DIRS:
    print(f"   {d}/")

Project directory structure created:
   data/raw/beijing/
   data/raw/jakarta/
   data/processed/
   models/saved_models/
   outputs/figures/
   outputs/tables/


In [7]:
# CELL 0.5 — Library Version Log
# Documents exact library versions for full reproducibility
# Anyone rerunning this notebook should match these versions
# ============================================================

import sklearn
import xgboost
import lightgbm
import shap
import statsmodels
import arch
import matplotlib
import seaborn
import plotly

version_log = {
    'python'        : __import__('sys').version.split()[0],
    'numpy'         : np.__version__,
    'pandas'        : pd.__version__,
    'statsmodels'   : statsmodels.__version__,
    'arch'          : arch.__version__,
    'scikit-learn'  : sklearn.__version__,
    'xgboost'       : xgboost.__version__,
    'lightgbm'      : lightgbm.__version__,
    'shap'          : shap.__version__,
    'matplotlib'    : matplotlib.__version__,
    'seaborn'       : seaborn.__version__,
    'plotly'        : plotly.__version__,
}

version_df = pd.DataFrame(
    version_log.items(),
    columns=['Library', 'Version']
)

print("Library Version Log")
print("=" * 35)
print(version_df.to_string(index=False))
print("=" * 35)
print(f"\nLogged at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# --- Save version log to outputs ---
version_df.to_csv('outputs/tables/version_log.csv', index=False)
print("Version log saved to outputs/tables/version_log.csv")

Library Version Log
     Library Version
      python 3.11.15
       numpy  1.26.4
      pandas   2.3.3
 statsmodels  0.14.6
        arch   8.0.0
scikit-learn   1.6.1
     xgboost   2.1.4
    lightgbm   4.6.0
        shap  0.48.0
  matplotlib  3.10.8
     seaborn  0.13.2
      plotly  5.24.1

Logged at: 2026-03-14 17:17:16
Version log saved to outputs/tables/version_log.csv


# **CHAPTER 1: Data Acquisition**

---
## Objective
Load all raw data from their original sources with full provenance documented.
No transformations, no cleaning — raw data only at this stage.

## Data Sources
| Dataset | Source | Coverage | Variables |
|---|---|---|---|
| Beijing Multi-Site Air Quality | UCI ML Repository (ID: 501) | Mar 2013 – Feb 2017, 12 stations | 6 pollutants + 6 meteorological |
| Jakarta Air Quality | OpenAQ REST API | 2022 – 2025 | PM2.5, PM10, NO2, O3, SO2 |

## What This Chapter Does
- `1.2` — Fetch Beijing dataset from UCI and inspect raw structure
- `1.3` — Extract and preview each of the 12 station DataFrames individually
- `1.4` — Fetch Jakarta data from OpenAQ API with documented endpoint
- `1.5` — Raw inventory: shapes, dtypes, column names for both datasets
- `1.6` — Station metadata documentation with coordinates and district context

> ⚠️ **No modifications are made to data in this chapter.**
> All cleaning, imputation, and transformation happens in Chapters 3 and 4.

In [8]:

# CELL 1.2 — Beijing Dataset Acquisition (UCI ML Repository)
# Fetches the official Beijing Multi-Site Air Quality dataset
# directly from UCI using the ucimlrepo package
# Dataset ID: 501 | DOI: 10.24432/C5RK5G
# ============================================================

print("Fetching Beijing Multi-Site Air Quality dataset from UCI...")
print("Dataset ID : 501")
print("Citation   : Zhang et al. (2017), Proc. Royal Society A")
print("-" * 60)

# --- Fetch from UCI ---
beijing_raw = fetch_ucirepo(id=501)

# --- Extract features and metadata ---
beijing_features  = beijing_raw.data.features
beijing_targets   = beijing_raw.data.targets
beijing_metadata  = beijing_raw.metadata
beijing_variables = beijing_raw.variables

# --- Combine features and target into one DataFrame ---
df_beijing_combined = pd.concat([beijing_features, beijing_targets], axis=1)

# --- Preview ---
print(f"\n✅ Beijing dataset fetched successfully.")
print(f"   Combined shape : {df_beijing_combined.shape}")
print(f"   Columns        : {list(df_beijing_combined.columns)}")
print(f"\n--- First 5 rows ---")
df_beijing_combined.head()

Fetching Beijing Multi-Site Air Quality dataset from UCI...
Dataset ID : 501
Citation   : Zhang et al. (2017), Proc. Royal Society A
------------------------------------------------------------


DatasetNotFoundError: "Beijing Multi-Site Air Quality" dataset (id=501) exists in the repository, but is not available for import. Please select a dataset from this list: https://archive.ics.uci.edu/datasets?skip=0&take=10&sort=desc&orderBy=NumHits&search=&Python=true

# **CHAPTER 2: Data Validation & Temporal Integrity**

# **CHAPTER 3: Missing Data Analysis**

# **CHAPTER 4: Preprocessing & Imputation**

# **CHAPTER 5: Exploratory Data Analysis**

# **CHAPTER 6: Stationarity & Time-Series Diagnostics**

# **CHAPTER 7: Feature Engineering**

# **CHAPTER 8: Train/Validation/Test Split**

# **CHAPTER 9: Baseline Models**

# **CHAPTER 10: Classical Statistical Models**

# **CHAPTER 11: Machine Learning Models**


# **CHAPTER 12: Uncertainty Quantification**

# **CHAPTER 13: Model Evaluation & Comparison**

# **CHAPTER 14: Interpretability**

# **CHAPTER 15: Regime & Structural Analysis**

# **CHAPTER 16: Risk Intelligence Framework**

# **CHAPTER 17: Jakarta Extension**

# **CHAPTER 18: Conclusions & Limitations**